# F1 Qualifying Monte Carlo Simulation

## Setup

### Packages

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import fastf1

### Setting up FastF1

Set up F1 data cache

In [ ]:
import os

# Create cache directory if it doesn't exist
file_path = 'C:\\Users\\stuwi\\OneDrive\\Documents\\F1 Monte Carlo sim'
cache_dir = os.path.expanduser(file_path)
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

# Enable the cache
fastf1.Cache.enable_cache(cache_dir)

# Check cache is correct
print(fastf1.Cache)

FastF1 cache (104.78 MB) C:\Users\stuwi\OneDrive\Documents\F1 Monte Carlo sim


## Functions

### Driver session stats

In [ ]:
def create_driver_stats(session):
    ''''
    Takes session as input.
    Filters to only competitive, legal laps (within 107% of fastest).
    Creates LapTimeSeconds column.
    Exports driver lap time mean, standard deviation and count.
    Drivers only with 1 lap in session have median std.
    '''

    laps = session.pick_quicklaps().loc[lambda df: ~df['Deleted']].copy()
    laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

    driver_stats = laps.groupby('Driver')['LapTimeSeconds'].aggregate(['mean', 'std', 'count'])

    backup_std = driver_stats['std'].median()
    driver_stats['std'] = driver_stats['std'].fillna(backup_std)

    return driver_stats

### Simulate qualifying

In [ ]:
def simulate_qualifying(driver_stats, n_attemps=2):
    '''
    Takes driver stats contain name, mean and std as input.
    Default number of attempts is 2, as is typical in Q3.
    Uses normal distribution with mean and std to generate a lap time.
    Returns list of drivers sorted by position, P1 first.
    '''

    results = {}

    for driver in driver_stats.index:
        mean = driver_stats.loc[driver, 'mean']
        std = driver_stats.loc[driver, 'std']

        lap_times = np.random.normal(mean, std, n_attemps)
        best_lap = lap_times.min()

        results[driver] = best_lap

    sorted_drivers = sorted(results, key=results.get)

    return sorted_drivers

### Repeated qualifying sim

In [ ]:
def qualifying_MC(driver_stats, n=500):
    '''
    Takes driver stats contain name, mean and std as input, and number of simulations, default 500.
    Returns dictionary containing drivers as keys, and values as dictionaries with counts on positions.
    '''

    from collections import defaultdict

    # Automically create position with count 0 if not seen before, with fresh dictionary for each driver.
    position_counts = defaultdict(lambda: defaultdict(int))

    for i in range(n):
        result = simulate_qualifying(driver_stats)

        # Add count for that position for driver, adjust for 0 indexing.
        for position, driver in enumerate(result):
            position_counts[driver][position + 1] +=1

    return position_counts

### Calculate probabilities for positions

In [ ]:
def get_position_probability(position_counts, n=500):
    '''
    Takes position counts in and returns counts as probabilities.
    '''
    position_probabilities = {}

    for driver, positions in position_counts.items():
        position_probabilities[driver] = {position : count / n 
                                          for position, count in position_counts[driver].items()}
    
    return position_probabilities

### Parent function

In [ ]:
def monte_carlo_qualifying(gp, year, n=500):
    '''
    Input is GP name, year, and number of simulations n.
    Returns dictionary of drivers with inner dictionary of probabilities for each position.
    '''
    if n < 1:
        raise ValueError('n must be a positive integer.')
    
    session = fastf1.get_session(year, gp, 'Q')
    session.load()

    q1, q2, q3 = session.laps.split_qualifying_sessions()

    driver_stats = create_driver_stats(q3)

    position_counts = qualifying_MC(driver_stats, n)
    position_probabilities = get_position_probability(position_counts, n)

    df = pd.DataFrame.from_dict(position_probabilities, orient='index')

    df = df.fillna(0)
    df.index.name = 'Driver'
    df.columns.name = 'Position'
    df = df.reindex(sorted(df.columns), axis=1)

    return df

In [180]:
monte_carlo_qualifying('Abu Dhabi', 2021, 1000)

core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['33', '44', '4', '11', '55', '77', '16', '22', '31', '3', '14', '10', '18', '99', '5', '6', '63', '7', '47', '9']


Position,1,2,3,4,5,6,7,8,9,10
Driver,,,,,,,,,,
HAM,0.552,0.429,0.017,0.001,0.001,0.000,0.000,0.000,0.000,0.000
VER,0.423,0.117,0.042,0.030,0.037,0.038,0.036,0.025,0.028,0.224
NOR,0.022,0.107,0.147,0.089,0.103,0.116,0.097,0.097,0.065,0.157
SAI,0.002,0.075,0.143,0.212,0.213,0.170,0.116,0.050,0.012,0.007
BOT,0.001,0.018,0.111,0.227,0.273,0.220,0.114,0.032,0.003,0.001
PER,0.000,0.227,0.446,0.261,0.064,0.002,0.000,0.000,0.000,0.000
LEC,0.000,0.020,0.056,0.097,0.145,0.184,0.205,0.130,0.079,0.084
TSU,0.000,0.007,0.038,0.083,0.159,0.249,0.265,0.128,0.049,0.022
RIC,0.000,0.000,0.000,0.000,0.001,0.009,0.041,0.159,0.358,0.432
